<a href="https://colab.research.google.com/github/kunalchauhann/DSVLAB/blob/main/BCSE1593_Data_Wrangling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BCSE 1593 – Data Science and Visualization Lab

## Practical: Data Wrangling

In this practical, we will take a retail transaction dataset and reorganise it into forms that are easier to analyse, compare and report.

The work will follow a simple sequence:

> **Load → understand the structure → create hierarchical indexes → combine tables → merge related data → reshape → verify → build a final report**

We will work with the **Supermart Grocery Sales – Retail Analytics Dataset**, a Kaggle retail dataset containing order, customer, category, region, date, sales, discount and profit information.

### In this practical, we will learn to

- create and work with a **MultiIndex**
- select records from specific index levels
- reorder and sort hierarchical indexes
- combine tables using `concat()`
- perform database-style joins using `merge()`
- merge tables using index columns
- use `join()` for index-based combination
- reshape data from wide to long using `melt()`
- reshape data from long to wide using `pivot()` and `pivot_table()`
- create and work with hierarchical columns
- use `stack()` and `unstack()`
- use `wide_to_long()` for repeated column patterns
- verify the structure after each major transformation

The focus is on understanding **how data changes shape** and choosing the correct pandas operation for the task.

# Part A – Getting Started

## 1. Importing pandas

Pandas will be our main tool for working with tabular data, indexes, joins and reshaping.

### Live coding

Import pandas using the standard alias `pd`.

In [57]:
import pandas as pd
print("Importing pandas")

# Import pandas using the standard alias pd.

Importing pandas


## 2. Importing NumPy

NumPy is useful for small helper tables and numerical operations during the practical.

### Live coding

Import NumPy using the standard alias `np`.

In [58]:
import numpy as np
print("Importing NumPy")

# Import NumPy using the standard alias np.

Importing NumPy


# Part B – Loading the Dataset

## 3. Dataset source – Supermart Grocery Sales

The dataset used here is **Supermart Grocery Sales – Retail Analytics Dataset**, available on Kaggle.

Kaggle source:

**https://www.kaggle.com/datasets/mohamedharris/supermart-grocery-sales-retail-analytics-dataset**

The dataset contains retail order records such as `Order ID`, `Customer Name`, `Category`, `Sub Category`, `City`, `Order Date`, `Region`, `Sales`, `Discount`, and `Profit`.

### Getting the file

Download **Supermart Grocery Sales - Retail Analytics Dataset.csv** from Kaggle and keep it in the same folder as this notebook.

The loading cell below first tries a local file. A public copy of the same CSV is provided as a fallback for practice environments where the local file is not available. **The dataset source remains Kaggle.**

In [59]:
file_name = "Supermart Grocery Sales - Retail Analytics Dataset.csv"

try:
    sales = pd.read_csv(file_name)
except FileNotFoundError:
    url = "https://raw.githubusercontent.com/godwinnwalozie/Supermart-Grocery-Sales---Retail-Analytics-Dataset/main/Supermart%20Grocery%20Sales%20-%20Retail%20Analytics%20Dataset.csv"
    sales = pd.read_csv(url)

print("Dataset loaded successfully")
sales.head()

Dataset loaded successfully


,Order ID,Customer Name,Category,Sub Category,City,Order Date,Region,Sales,Discount,Profit,State
0,OD1,Harish,Oil & Masala,Masalas,Vellore,11-08-2017,North,1254,0.12,401.28,Tamil Nadu
1,OD2,Sudha,Beverages,Health Drinks,Krishnagiri,11-08-2017,South,749,0.18,149.80,Tamil Nadu
2,OD3,Hussain,Food Grains,Atta & Flour,Perambalur,06-12-2017,West,2360,0.21,165.20,Tamil Nadu
3,OD4,Jackson,Fruits & Veggies,Fresh Vegetables,Dharmapuri,10-11-2016,South,896,0.25,89.60,Tamil Nadu
4,OD5,Ridhesh,Food Grains,Organic Staples,Ooty,10-11-2016,South,2355,0.26,918.45,Tamil Nadu


## 4. First look at the data

Before changing the structure, first understand what a row represents and which columns are available.

Check the shape, column names, data types and a quick statistical summary.

In [60]:
# Display the number of rows and columns.
print(f"Shape: {sales.shape}")
# Display the column names.
print(f"Columns: {sales.columns.tolist()}")
# Display the data types.
print(sales.dtypes)
# Display the first five rows.
print(sales.head())
# Display the numerical summary.
print(sales.describe())

Shape: (9994, 11)
Columns: ['Order ID', 'Customer Name', 'Category', 'Sub Category', 'City', 'Order Date', 'Region', 'Sales', 'Discount', 'Profit', 'State']
Order ID          object
Customer Name     object
Category          object
Sub Category      object
City              object
Order Date        object
Region            object
Sales              int64
Discount         float64
Profit           float64
State             object
dtype: object
  Order ID Customer Name          Category      Sub Category         City  \
0      OD1        Harish      Oil & Masala           Masalas      Vellore   
1      OD2         Sudha         Beverages     Health Drinks  Krishnagiri   
2      OD3       Hussain       Food Grains      Atta & Flour   Perambalur   
3      OD4       Jackson  Fruits & Veggies  Fresh Vegetables   Dharmapuri   
4      OD5       Ridhesh       Food Grains   Organic Staples         Ooty   

   Order Date Region  Sales  Discount  Profit       State  
0  11-08-2017  North   1254    

## 5. Prepare the date column

Wrangling often becomes easier after dates are stored as real datetime values. This lets us create year, month and other time-based fields later.

### Function used

`pd.to_datetime()` converts text values into pandas datetime values.

The argument `errors='coerce'` turns an unrecognised date into `NaT` instead of stopping the program.

In [61]:
# Convert Order Date to datetime.
sales['Order Date'] = pd.to_datetime(sales['Order Date'], errors='coerce')

# Check the dtype of Order Date.
sales['Order Date'].dtype

dtype('<M8[ns]')

## 6. Create a working copy

Keep the original table unchanged and perform wrangling on a working copy. This is useful when you need to compare the raw and transformed versions later.

In [62]:
# Create a working copy named wrangled.
wrangled = sales.copy()
# Display the shape of wrangled.
print(wrangled.shape)

(9994, 11)


# Part C – Hierarchical Indexing (MultiIndex)

## 7. Why use a hierarchical index?

A normal DataFrame uses one index for rows. A **MultiIndex** uses two or more levels.

For example, retail data can naturally be organised as:

> **Region → Category → records**

This is useful when the same type of question is repeatedly asked at more than one level, such as regional and category-wise reporting.

### Why is this useful in practice?

A sales team may want to move from **Region** to **Category** without creating separate tables for every combination. A MultiIndex keeps that relationship organised in one structure and makes repeated selections and summaries easier.

### Main functions

- `set_index()` – move columns into the index
- `pd.MultiIndex.from_arrays()` – create levels directly
- `pd.MultiIndex.from_tuples()` – create levels from tuples
- `.loc[]` – select using index labels
- `xs()` – select a cross-section
- `swaplevel()` – exchange level positions
- `reorder_levels()` – choose a new level order
- `sort_index()` – sort the index
- `reset_index()` – return index levels to columns


## 8. Creating a MultiIndex with `set_index()`

Move `Region` and `Category` into the index.

The original columns are not lost; they are simply moved into the row index.

This creates a structure such as **Region → Category → rows**.

In [63]:
# Keep a few useful columns for the MultiIndex example.
mi_data = wrangled[['Order ID', 'Customer Name', 'Region', 'Category', 'Sales', 'Profit']].copy()

# Set Region and Category as the two index levels.
region_category_mi = mi_data.set_index(['Region', 'Category'])
# Display the first few rows.
print(region_category_mi.head())

                        Order ID Customer Name  Sales  Profit
Region Category                                              
North  Oil & Masala          OD1        Harish   1254  401.28
South  Beverages             OD2         Sudha    749  149.80
West   Food Grains           OD3       Hussain   2360  165.20
South  Fruits & Veggies      OD4       Jackson    896   89.60
       Food Grains           OD5       Ridhesh   2355  918.45


## 9. Inspecting the index levels

A MultiIndex contains multiple named levels. Inspecting them helps you understand how the table is organised.

Useful attributes and methods:

- `.index.names` – level names
- `.index.get_level_values()` – values from one level
- `.index.nlevels` – number of levels

In [64]:
# Display the names of the index levels.
print(region_category_mi.index.names)
# Display the number of index levels.
print(region_category_mi.index.nlevels)
# Display unique Region values from the first level.
print(region_category_mi.index.get_level_values(level='Region').unique().to_list())
# Display unique Category values from the second level.
print(region_category_mi.index.get_level_values(level='Category').unique().to_list())

['Region', 'Category']
2
['North', 'South', 'West', 'Central', 'East']
['Oil & Masala', 'Beverages', 'Food Grains', 'Fruits & Veggies', 'Bakery', 'Snacks', 'Eggs, Meat & Fish']


## 10. Selecting from a MultiIndex with `.loc`

When the DataFrame has meaningful row labels, `.loc` can select using those labels.

Here we will use the first available Region and Category from the data so that the example remains valid even if a student uses a filtered copy of the dataset.

In [99]:
# Pick one Region and one Category from the available values.
region_value = region_category_mi.index.get_level_values('Region').unique()[0]
category_value = region_category_mi.index.get_level_values('Category').unique()[0]

# Select rows matching both levels with .loc.
print(region_category_mi.loc[(region_value, category_value)])

                    Order ID Customer Name  Sales  Profit
Region Category                                          
North  Oil & Masala      OD1        Harish   1254  401.28


/tmp/ipykernel_7240/967069324.py:6: PerformanceWarning: indexing past lexsort depth may impact performance.
  print(region_category_mi.loc[(region_value, category_value)])


## 11. Selecting one level with `xs()`

The `xs()` method means **cross-section**. It is useful when you want to select one level without writing the complete index key.

Example uses include:

- all categories in one Region
- all Regions for one Category

In [66]:
# Select one Region using xs().
print(region_category_mi.xs('South',level='Region'))
# Select one Category using xs(level='Category').
print(region_category_mi.xs(category_value,level='Category'))

                 Order ID Customer Name  Sales   Profit
Category                                               
Beverages             OD2         Sudha    749   149.80
Fruits & Veggies      OD4       Jackson    896    89.60
Food Grains           OD5       Ridhesh   2355   918.45
Snacks               OD13        Sharon   1659   315.21
Snacks               OD44        Mathew   2311  1039.95
...                   ...           ...    ...      ...
Fruits & Veggies   OD9938         Yusuf   1610   370.30
Snacks             OD9939         Vince   1480   148.00
Snacks             OD9940        Adavan   1560   624.00
Beverages          OD9941           Anu   2379   428.22
Snacks             OD9942        Amrish   1843   829.35

[1619 rows x 4 columns]
        Order ID Customer Name  Sales  Profit
Region                                       
North        OD1        Harish   1254  401.28
Central     OD16        Ramesh   1440  100.80
East        OD30          Shah    989   69.23
East        OD33 

## 12. Creating a MultiIndex explicitly with `from_arrays()`

A MultiIndex does not have to come from existing DataFrame columns. It can also be created directly from arrays.

This is useful when building small lookup tables or preparing a report structure.

In [67]:
regions = ['North', 'South', 'West', 'East']
categories = ['Bakery', 'Beverages', 'Snacks', 'Oil & Masala']
sales_values = [12000, 9500, 8300, 10600]

# Create a MultiIndex from the two arrays.
Mi_from_arrays = pd.MultiIndex.from_arrays([regions, categories])
# Display the resulting MultiIndex.
print(Mi_from_arrays)
# Give the levels the names Region and Category.

# Build a small DataFrame named regional_sales.
regional_sales = pd.DataFrame({'Sales': sales_values}, index=Mi_from_arrays)
print(regional_sales)

MultiIndex([('North',       'Bakery'),
            ('South',    'Beverages'),
            ( 'West',       'Snacks'),
            ( 'East', 'Oil & Masala')],
           )
                    Sales
North Bakery        12000
South Beverages      9500
West  Snacks         8300
East  Oil & Masala  10600


## 13. Creating a MultiIndex with `from_tuples()`

`from_tuples()` is convenient when each index key is naturally written as a pair such as `(Region, Category)`.

In [68]:
keys = [('North', 'Bakery'), ('South', 'Beverages'), ('West', 'Snacks'), ('East', 'Oil & Masala')]

# Create a MultiIndex from the tuple pairs.
Mi_from_tuples = pd.MultiIndex.from_tuples(keys,names=['Region','Category'])
# Display the resulting MultiIndex.
print(Mi_from_tuples)

MultiIndex([('North',       'Bakery'),
            ('South',    'Beverages'),
            ( 'West',       'Snacks'),
            ( 'East', 'Oil & Masala')],
           names=['Region', 'Category'])


## 14. Reordering the index levels

The order of levels affects how the data is presented and selected.

Two useful functions are:

- `swaplevel()` – quickly exchange two levels
- `reorder_levels()` – specify the complete new order

In [69]:
# Swap Region and Category.
swapped=region_category_mi.swaplevel()

# Reorder the levels so Category comes before Region.
reordered=region_category_mi.reorder_levels(['Category','Region'])
# Display both results.
print(swapped)
print(reordered)

                         Order ID Customer Name  Sales  Profit
Category          Region                                      
Oil & Masala      North       OD1        Harish   1254  401.28
Beverages         South       OD2         Sudha    749  149.80
Food Grains       West        OD3       Hussain   2360  165.20
Fruits & Veggies  South       OD4       Jackson    896   89.60
Food Grains       South       OD5       Ridhesh   2355  918.45
...                           ...           ...    ...     ...
Eggs, Meat & Fish West     OD9990        Sudeep    945  359.10
Bakery            West     OD9991          Alan   1195   71.70
Food Grains       West     OD9992          Ravi   1567  501.44
Oil & Masala      West     OD9993          Peer   1659  597.24
Food Grains       West     OD9994        Ganesh   1034  165.44

[9994 rows x 4 columns]
                         Order ID Customer Name  Sales  Profit
Category          Region                                      
Oil & Masala      North       

## 15. Sorting by index levels

After changing index levels, sorting keeps the hierarchy organised.

Use `sort_index()` for MultiIndex sorting. You can sort all levels or specify a particular level.

In [70]:
# Set names for the index levels so they can be referenced by name
regional_sales.index.names = ['Region', 'Category']

# Sort regional_sales by all index levels.
print(regional_sales.sort_index())

# Sort another version primarily by Category.
print(regional_sales.sort_index(level='Category'))

                     Sales
Region Category           
East   Oil & Masala  10600
North  Bakery        12000
South  Beverages      9500
West   Snacks         8300
                     Sales
Region Category           
North  Bakery        12000
South  Beverages      9500
East   Oil & Masala  10600
West   Snacks         8300


## 16. Selecting a range with `IndexSlice`

`pd.IndexSlice` makes multi-level `.loc[]` expressions easier to read. It is helpful when you want several values from one level while keeping all values from another level.

In [71]:
# Create an IndexSlice object.
idx = pd.IndexSlice

# Use idx with .loc to select multiple Regions from region_category_mi.
# Keep all Category values.
print(region_category_mi.loc[idx[:,['Bakery','Beverages']],:])

                  Order ID Customer Name  Sales  Profit
Region  Category                                       
West    Bakery         OD9         Hafiz    791  181.93
        Bakery        OD10      Krithika   1795  484.65
Central Bakery        OD17          Alan   1678  318.82
East    Bakery        OD28       Jackson    979  401.39
Central Bakery        OD45        Amrish   1663  731.72
...                    ...           ...    ...     ...
West    Beverages   OD9943        Roshan   2088  668.16
        Beverages   OD9946          Peer   2400  336.00
        Beverages   OD9954         Veena   1693  304.74
        Beverages   OD9976        Mathew   1885  414.70
        Beverages   OD9978           Anu   2028  628.68

[2813 rows x 4 columns]


## 17. Resetting a MultiIndex

A hierarchical index is useful for analysis, but ordinary columns are often easier for merging, exporting and further transformation.

`reset_index()` moves index levels back into columns.

In [72]:
# Reset the MultiIndex.
region_category_mi.reset_index()
# Display the resulting columns.
region_category_mi.reset_index().columns

Index(['Region', 'Category', 'Order ID', 'Customer Name', 'Sales', 'Profit'], dtype='object')

## 18. MultiIndex created automatically by `groupby()`

Grouping by more than one column commonly creates a MultiIndex when `as_index=True`. This is one of the most useful real-world ways to encounter hierarchical indexes.

In [100]:
# Create a Region-Category summary.
# Calculate total Sales, total Profit and average Discount.
# Keep Region and Category as index levels.
region_category_summary=wrangled.groupby(['Region','Category'],as_index=True).agg(Total_Sales=('Sales','sum'),
                                                                                  Total_Profit=('Profit','sum'),
                                                                                  Average_Discount=('Discount','mean'))

# Display the summary.
region_category_summary
# Display the summary.

Total_Sales  Total_Profit  Average_Discount
Region  Category                                                      
Central Bakery                  448343     109543.91          0.231294
        Beverages               501194     128017.14          0.231749
        Eggs, Meat & Fish       516642     126587.35          0.232356
        Food Grains             463683     111517.05          0.223613
        Fruits & Veggies        526507     131699.53          0.232676
        Oil & Masala            453293     109261.91          0.224900
        Snacks                  558494     140179.95          0.224187
East    Bakery                  588241     153741.06          0.227908
        Beverages               577206     144498.17          0.230000
        Eggs, Meat & Fish       677830     164465.08          0.231180
        Food Grains             598868     154932.01          0.228931
        Fruits & Veggies        564237     141292.78          0.227083
        Oil & Masala            622071     155569.22          0.222644
        Snacks                  619915     159847.26          0.225944
North   Oil & Masala              1254        401.28          0.120000
South   Bakery                  361385      89102.07          0.223333
        Beverages               344666      84058.78          0.231938
        Eggs, Meat & Fish       360963      93728.18          0.230258
        Food Grains             353069      94823.55          0.233750
        Fruits & Veggies        352162      92961.26          0.228376
        Oil & Masala            298859      73909.18          0.228218
        Snacks                  369357      94979.87          0.213068
West    Bakery                  714312     176134.02          0.220593
        Beverages               662247     169031.67          0.228697
        Eggs, Meat & Fish       711966     182576.61          0.220231
        Food Grains             699652     167890.03          0.228920
        Fruits & Veggies        657821     164446.81          0.228989
        Oil & Masala            662965     158753.70          0.224593
        Snacks                  689780     173171.77          0.221569

# Part D – Combining Data Sets

## 19. Why combine tables?

Real projects rarely keep every piece of information in one table. You may receive files by month, by branch, or by business unit. Pandas provides several ways to bring them together.

### Why is this useful in practice?

A company may receive January, February and March sales files separately. The analyst needs one table before calculating totals, trends or reports. The correct combining method depends on whether the records should be **stacked**, **matched by a key**, or **aligned by index**.

### Main function

`pd.concat()` combines objects along rows or columns.

Use:

- `axis=0` for row-wise combination
- `axis=1` for column-wise combination
- `keys=` when you want to keep the source of each block


## 20. Preparing two related tables

To understand concatenation clearly, split the transaction table into two smaller batches. The records remain the same; only the storage is different.

In [101]:
# Create one table containing the first 5 rows.
batch_1 = wrangled.head(5).copy()

# Create another table containing the next 5 rows.
batch_2 = wrangled.iloc[5:10].copy()

# Display both batches.
display(batch_1)
display(batch_2)

,Order ID,Customer Name,Category,Sub Category,City,Order Date,Region,Sales,Discount,Profit,State
0,OD1,Harish,Oil & Masala,Masalas,Vellore,2017-11-08,North,1254,0.12,401.28,Tamil Nadu
1,OD2,Sudha,Beverages,Health Drinks,Krishnagiri,2017-11-08,South,749,0.18,149.80,Tamil Nadu
2,OD3,Hussain,Food Grains,Atta & Flour,Perambalur,2017-06-12,West,2360,0.21,165.20,Tamil Nadu
3,OD4,Jackson,Fruits & Veggies,Fresh Vegetables,Dharmapuri,2016-10-11,South,896,0.25,89.60,Tamil Nadu
4,OD5,Ridhesh,Food Grains,Organic Staples,Ooty,2016-10-11,South,2355,0.26,918.45,Tamil Nadu


,Order ID,Customer Name,Category,Sub Category,City,Order Date,Region,Sales,Discount,Profit,State
5,OD6,Adavan,Food Grains,Organic Staples,Dharmapuri,2015-06-09,West,2305,0.26,322.70,Tamil Nadu
6,OD7,Jonas,Fruits & Veggies,Fresh Vegetables,Trichy,2015-06-09,West,826,0.33,346.92,Tamil Nadu
7,OD8,Hafiz,Fruits & Veggies,Fresh Fruits,Ramanadhapuram,2015-06-09,West,1847,0.32,147.76,Tamil Nadu
8,OD9,Hafiz,Bakery,Biscuits,Tirunelveli,2015-06-09,West,791,0.23,181.93,Tamil Nadu
9,OD10,Krithika,Bakery,Cakes,Chennai,2015-06-09,West,1795,0.27,484.65,Tamil Nadu


## 21. Vertical concatenation with `concat()`

When two tables have the same columns and represent the same kind of record, append one underneath the other with `pd.concat()`.

**Practical use:** this is common when monthly, branch-wise or batch-wise files have the same structure and need to become one analysis table.


In [102]:
# Concatenate batch_1 and batch_2 row-wise.
combined_rows = pd.concat([batch_1, batch_2], axis=0, ignore_index=True)

# Display the combined table.
combined_rows

,Order ID,Customer Name,Category,Sub Category,City,Order Date,Region,Sales,Discount,Profit,State
0,OD1,Harish,Oil & Masala,Masalas,Vellore,2017-11-08,North,1254,0.12,401.28,Tamil Nadu
1,OD2,Sudha,Beverages,Health Drinks,Krishnagiri,2017-11-08,South,749,0.18,149.80,Tamil Nadu
2,OD3,Hussain,Food Grains,Atta & Flour,Perambalur,2017-06-12,West,2360,0.21,165.20,Tamil Nadu
3,OD4,Jackson,Fruits & Veggies,Fresh Vegetables,Dharmapuri,2016-10-11,South,896,0.25,89.60,Tamil Nadu
4,OD5,Ridhesh,Food Grains,Organic Staples,Ooty,2016-10-11,South,2355,0.26,918.45,Tamil Nadu
5,OD6,Adavan,Food Grains,Organic Staples,Dharmapuri,2015-06-09,West,2305,0.26,322.70,Tamil Nadu
6,OD7,Jonas,Fruits & Veggies,Fresh Vegetables,Trichy,2015-06-09,West,826,0.33,346.92,Tamil Nadu
7,OD8,Hafiz,Fruits & Veggies,Fresh Fruits,Ramanadhapuram,2015-06-09,West,1847,0.32,147.76,Tamil Nadu
8,OD9,Hafiz,Bakery,Biscuits,Tirunelveli,2015-06-09,West,791,0.23,181.93,Tamil Nadu
9,OD10,Krithika,Bakery,Cakes,Chennai,2015-06-09,West,1795,0.27,484.65,Tamil Nadu


## 22. Concatenation with source labels

`keys=` adds another index level showing where each block came from. This creates a useful hierarchical structure while combining data.

In [103]:
# Concatenate the two batches with source labels Batch_1 and Batch_2.
combined_with__source=pd.concat([batch_1,batch_2],axis=0,keys=['Batch_1','Batch_2'])

# Display the result and inspect its MultiIndex.
combined_with__source

Order ID Customer Name          Category      Sub Category  \
Batch_1 0      OD1        Harish      Oil & Masala           Masalas   
        1      OD2         Sudha         Beverages     Health Drinks   
        2      OD3       Hussain       Food Grains      Atta & Flour   
        3      OD4       Jackson  Fruits & Veggies  Fresh Vegetables   
        4      OD5       Ridhesh       Food Grains   Organic Staples   
Batch_2 5      OD6        Adavan       Food Grains   Organic Staples   
        6      OD7         Jonas  Fruits & Veggies  Fresh Vegetables   
        7      OD8         Hafiz  Fruits & Veggies      Fresh Fruits   
        8      OD9         Hafiz            Bakery          Biscuits   
        9     OD10      Krithika            Bakery             Cakes   

                     City Order Date Region  Sales  Discount  Profit  \
Batch_1 0         Vellore 2017-11-08  North   1254      0.12  401.28   
        1     Krishnagiri 2017-11-08  South    749      0.18  149.80   
        2      Perambalur 2017-06-12   West   2360      0.21  165.20   
        3      Dharmapuri 2016-10-11  South    896      0.25   89.60   
        4            Ooty 2016-10-11  South   2355      0.26  918.45   
Batch_2 5      Dharmapuri 2015-06-09   West   2305      0.26  322.70   
        6          Trichy 2015-06-09   West    826      0.33  346.92   
        7  Ramanadhapuram 2015-06-09   West   1847      0.32  147.76   
        8     Tirunelveli 2015-06-09   West    791      0.23  181.93   
        9         Chennai 2015-06-09   West   1795      0.27  484.65   

                State  
Batch_1 0  Tamil Nadu  
        1  Tamil Nadu  
        2  Tamil Nadu  
        3  Tamil Nadu  
        4  Tamil Nadu  
Batch_2 5  Tamil Nadu  
        6  Tamil Nadu  
        7  Tamil Nadu  
        8  Tamil Nadu  
        9  Tamil Nadu

## 23. Horizontal concatenation

Use `axis=1` when the two objects contain different columns that refer to the same rows. In this case, we will align two small tables by their reset index.

**Practical use:** this is useful when two reports describe the same observations but contain different attributes and need to be placed side-by-side.


In [104]:
left_part = wrangled[['Order ID', 'Customer Name']].head(8).reset_index(drop=True)
right_part = wrangled[['Category', 'Region', 'Sales']].head(8).reset_index(drop=True)

# Concatenate the two tables side-by-side.
horizontal = pd.concat([left_part, right_part], axis=1)

# Display the result.
horizontal

,Order ID,Customer Name,Category,Region,Sales
0,OD1,Harish,Oil & Masala,North,1254
1,OD2,Sudha,Beverages,South,749
2,OD3,Hussain,Food Grains,West,2360
3,OD4,Jackson,Fruits & Veggies,South,896
4,OD5,Ridhesh,Food Grains,South,2355
5,OD6,Adavan,Food Grains,West,2305
6,OD7,Jonas,Fruits & Veggies,West,826
7,OD8,Hafiz,Fruits & Veggies,West,1847


# Part E – Database-Style Joins with `merge()`

## 24. Why use `merge()`?

A database-style join combines related tables through one or more key columns. Instead of simply stacking rows, it matches records that share a key.

### Why is this useful in practice?

A retail company may keep **sales transactions** in one table and **customer master details** in another. To prepare a customer report, we match the two tables using `Customer Name` or another customer ID.

### Main arguments

- `on=` – same key name in both tables
- `left_on=` / `right_on=` – different key names
- `how=` – `inner`, `left`, `right`, or `outer`
- `indicator=True` – show where each record came from
- `validate=` – check whether the relationship matches the expected join type


## 25. Preparing two different related tables

We will use two tables with **different purposes and different columns**:

- **`customer_summary`** – a transaction-based performance table containing total sales and profit for each customer.
- **`customer_master`** – a separate customer lookup table containing region and customer tier. It also contains a couple of customers who do not yet have transactions, so that unmatched join results are visible.

This is closer to a real reporting workflow than merging two summaries created from exactly the same columns.


In [78]:
# Customer performance table from transaction records.
customer_summary = (
    wrangled.groupby('Customer Name', as_index=False)
           .agg(Total_Sales=('Sales', 'sum'),
                Total_Profit=('Profit', 'sum'))
)

# Separate customer master / lookup table.
# It contains customer attributes rather than sales measures.
master_base = (
    wrangled[['Customer Name', 'Region']]
    .drop_duplicates('Customer Name')
    .sort_values('Customer Name')
    .head(7)
    .copy()
)
master_base['Customer Tier'] = ['Gold', 'Silver', 'Gold', 'Standard', 'Silver', 'Gold', 'Standard'][:len(master_base)]

extra_customers = pd.DataFrame({
    'Customer Name': ['New Customer A', 'New Customer B'],
    'Region': ['East', 'West'],
    'Customer Tier': ['Gold', 'Standard']
})

customer_master = pd.concat([master_base, extra_customers], ignore_index=True)

print('Customer performance table:')
print(customer_summary.head())
print('\nSeparate customer master table:')
print(customer_master)


Customer performance table:
  Customer Name  Total_Sales  Total_Profit
0        Adavan       315341      78044.29
1         Aditi       299674      78439.05
2         Akash       307488      82121.26
3          Alan       294397      74410.23
4        Amrish       333351      80191.89

Separate customer master table:
    Customer Name   Region Customer Tier
0          Adavan     West          Gold
1           Aditi     West        Silver
2           Akash     East          Gold
3            Alan  Central      Standard
4          Amrish  Central        Silver
5             Amy     West          Gold
6             Anu     East      Standard
7  New Customer A     East          Gold
8  New Customer B     West      Standard


## 26. Inner join

An **inner join** keeps only keys that appear in both tables. In this practical, it keeps customers who have both a transaction summary and a matching customer-master record.

**Practical use:** use an inner join when the report should contain only complete matches between two sources.


In [79]:
# Keep only customers present in both the transaction summary and customer master.
inner_joined = pd.merge(
    customer_summary,
    customer_master,
    on='Customer Name',
    how='inner'
)

print(inner_joined)


  Customer Name  Total_Sales  Total_Profit   Region Customer Tier
0        Adavan       315341      78044.29     West          Gold
1         Aditi       299674      78439.05     West        Silver
2         Akash       307488      82121.26     East          Gold
3          Alan       294397      74410.23  Central      Standard
4        Amrish       333351      80191.89  Central        Silver
5           Amy       301558      79159.94     West          Gold
6           Anu       283735      70238.28     East      Standard


## 27. Left join vs Right join – the side matters

The difference is **which table you choose to protect**.

- **Left join:** keep every row from the left table and add matches from the right table.
- **Right join:** keep every row from the right table and add matches from the left table.

Here, the **left table is `customer_summary`** and the **right table is `customer_master`**.

**Practical use:**

- Use a **left join** when your main report is the transaction/customer-performance table and you do not want to lose customers who have sales.
- Use a **right join** when the master table is the list you want to preserve, such as checking all registered customers even if some have no transactions.


In [80]:
# LEFT JOIN – keep all customers who appear in the sales summary.
left_joined = pd.merge(
    customer_summary,
    customer_master,
    on='Customer Name',
    how='left'
)

print('LEFT JOIN – customers missing from the master table:')
print(left_joined[left_joined['Customer Tier'].isna()])

# RIGHT JOIN – keep all customers in the customer master table.
right_joined = pd.merge(
    customer_summary,
    customer_master,
    on='Customer Name',
    how='right'
)

print('\nRIGHT JOIN – customers with no transaction summary:')
print(right_joined[right_joined['Total_Sales'].isna()])


LEFT JOIN – customers missing from the master table:
   Customer Name  Total_Sales  Total_Profit Region Customer Tier
7         Arutra       325720      87572.40    NaN           NaN
8         Arvind       293802      71767.23    NaN           NaN
9         Esther       283656      69588.12    NaN           NaN
10        Ganesh       278349      67094.89    NaN           NaN
11         Hafiz       251965      64037.35    NaN           NaN
12        Harish       293839      79107.83    NaN           NaN
13       Haseena       298924      73167.14    NaN           NaN
14       Hussain       307337      72354.97    NaN           NaN
15       Jackson       271793      70268.59    NaN           NaN
16         James       305915      71926.88    NaN           NaN
17         Jonas       293489      73497.19    NaN           NaN
18         Komal       295671      76165.17    NaN           NaN
19      Krithika       334361      85633.03    NaN           NaN
20         Kumar       262623      67

## 28. Outer join

An **outer join** keeps keys from both tables. It is useful when you want to audit all matched and unmatched records from both sources.

**Practical use:** this is useful during data reconciliation when you need to find records that exist in one system but not the other.


In [81]:
# Keep every customer appearing in either table.
outer_joined = pd.merge(
    customer_summary,
    customer_master,
    on='Customer Name',
    how='outer',
    indicator=True
)

print(outer_joined)


     Customer Name  Total_Sales  Total_Profit   Region Customer Tier  \
0           Adavan     315341.0      78044.29     West          Gold   
1            Aditi     299674.0      78439.05     West        Silver   
2            Akash     307488.0      82121.26     East          Gold   
3             Alan     294397.0      74410.23  Central      Standard   
4           Amrish     333351.0      80191.89  Central        Silver   
5              Amy     301558.0      79159.94     West          Gold   
6              Anu     283735.0      70238.28     East      Standard   
7           Arutra     325720.0      87572.40      NaN           NaN   
8           Arvind     293802.0      71767.23      NaN           NaN   
9           Esther     283656.0      69588.12      NaN           NaN   
10          Ganesh     278349.0      67094.89      NaN           NaN   
11           Hafiz     251965.0      64037.35      NaN           NaN   
12          Harish     293839.0      79107.83      NaN          

## 29. Merging when key names are different

Real datasets often use different names for the same key. Use `left_on=` and `right_on=` instead of forcing the column names to match first.

**Practical use:** one system may use `Customer Name` while another uses `Customer`. The data can still be combined without renaming the original field.


In [82]:
# Rename the key in the master table only for this example.
customer_master_alt = customer_master.rename(columns={'Customer Name': 'Customer'})

# Merge using different key names.
different_key_merge = pd.merge(
    customer_summary,
    customer_master_alt,
    left_on='Customer Name',
    right_on='Customer',
    how='left'
)

different_key_merge.head()


,Customer Name,Total_Sales,Total_Profit,Customer,Region,Customer Tier
0,Adavan,315341,78044.29,Adavan,West,Gold
1,Aditi,299674,78439.05,Aditi,West,Silver
2,Akash,307488,82121.26,Akash,East,Gold
3,Alan,294397,74410.23,Alan,Central,Standard
4,Amrish,333351,80191.89,Amrish,Central,Silver


## 30. Checking the type of merge

The `validate=` argument can check whether the relationship between tables matches your expectation.

For example, `many_to_one` expects repeated keys on the left but a single matching key on the right. That is appropriate here because many transaction rows may belong to one customer, while the customer master contains one record per customer.

**Practical use:** `validate=` helps catch accidental duplicate master records that could multiply rows in a report.


In [83]:
# Check a true many-to-one relationship:
# many transaction rows can belong to one customer-master row.
validated_merge = pd.merge(
    wrangled[['Order ID', 'Customer Name', 'Sales']],
    customer_master,
    on='Customer Name',
    how='left',
    validate='many_to_one'
)

validated_merge.head()


,Order ID,Customer Name,Sales,Region,Customer Tier
0,OD1,Harish,1254,NaN,NaN
1,OD2,Sudha,749,NaN,NaN
2,OD3,Hussain,2360,NaN,NaN
3,OD4,Jackson,896,NaN,NaN
4,OD5,Ridhesh,2355,NaN,NaN


## 31. Tracking unmatched rows with `indicator=True`

When auditing two tables, `indicator=True` creates a column showing whether each row came from the left table, right table, or both.

**Practical use:** this is helpful during reconciliation—for example, when checking which customers exist in the transaction system but are missing from the master system.


In [84]:
# Perform an outer merge with indicator=True.
merge_check = pd.merge(
    customer_summary,
    customer_master,
    on='Customer Name',
    how='outer',
    indicator=True
)

print(merge_check['_merge'].value_counts())


_merge
left_only     43
both           7
right_only     2
Name: count, dtype: int64


## 32. Merging on indexes

Sometimes the key is already stored as the index. `merge()` can work directly with index values by using `left_index=True` and `right_index=True`.

In practice, a regional sales table might be matched with a separate regional target table using the region stored as the index.


In [85]:
# Actual sales from transaction records.
region_sales = wrangled.groupby('Region')['Sales'].sum().to_frame('Actual_Sales')

# Separate business target table.
region_targets = pd.DataFrame({
    'Sales_Target': [18000, 17000, 16000, 15000, 19000],
    'Region_Manager': ['Anil', 'Meera', 'Rohan', 'Priya', 'Karan']
}, index=['Central', 'East', 'North', 'South', 'West'])

# Merge the two different tables using their indexes.
region_report_merge = pd.merge(
    region_sales,
    region_targets,
    left_index=True,
    right_index=True,
    how='left'
)

region_report_merge


,Actual_Sales,Sales_Target,Region_Manager
Region,,,
Central,3468156,18000,Anil
East,4248368,17000,Meera
North,1254,16000,Rohan
South,2440461,15000,Priya
West,4798743,19000,Karan


## 33. Using `join()` for index-based combination

`DataFrame.join()` is convenient when the joining key is already the index. It can make index-based combinations shorter and easier to read.

**Practical use:** once regional targets and actuals are indexed by `Region`, `join()` is often a readable way to build a management report.


In [86]:
# Create another separate lookup table indexed by Region.
region_manager = region_targets[['Region_Manager']]

# Join actual sales to the regional manager lookup.
region_report = region_sales.join(region_manager, how='left')

region_report


,Actual_Sales,Region_Manager
Region,,
Central,3468156,Anil
East,4248368,Meera
North,1254,Rohan
South,2440461,Priya
West,4798743,Karan


## 34. Choosing `merge()` or `join()`

Use `merge()` when:

- the key is stored in ordinary columns
- the left and right key names are different
- you need detailed control over join types

Use `join()` when:

- the key is already the index
- you want a concise index-based combination

**Practical rule:** first ask whether the relationship is a **column-to-column match** or an **index-to-index alignment**. That usually tells you which method to choose.


# Part F – Reshaping and Pivoting

## 35. Why reshape data?

The same information can be stored in different layouts. Wrangling often means changing the layout without changing the underlying business meaning.

### Why is this useful in practice?

A detailed sales table may be best for analysis, while a manager may want **Region as rows and Category as columns** for a report. Reshaping lets the same data move between these forms without rebuilding the data from scratch.


## 36. Create a wide-format sales table

We will first summarise sales by Region and Category. Then we will place categories across columns to create a wide layout.

In [87]:
# Summarise total Sales by Region and Category.
region_category_sales = (
    wrangled.groupby(['Region', 'Category'], as_index=False)['Sales']
           .sum()
)

# Create a wide table with Region as rows and Category as columns.
wide_sales = region_category_sales.pivot(
    index='Region', columns='Category', values='Sales'
)

# Display the wide table.

## 37. Wide-to-long with `melt()`

`melt()` turns multiple value columns into two columns: one column containing the variable name and another containing the value.

**Practical use:** `melt()` is useful when a report is stored in a wide layout but you need a long, tidy layout for grouping, plotting or further analysis.


In [88]:
# Convert wide_sales back to long format.
long_sales = wide_sales.reset_index().melt(
    id_vars='Region',
    var_name='Category',
    value_name='Sales'
)

# Display the result.

## 38. Inspecting the long-format result

After reshaping, always verify the row count, column names and missing values. Reshaping can introduce `NaN` where a Region does not contain a particular category.

In [89]:
# Display the shape of long_sales.

# Display the column names.

# Count missing Sales values.

## 39. Long-to-wide with `pivot()`

`pivot()` reverses a simple melt when each combination of index and column values identifies one observation.

**Practical use:** `pivot()` is useful when you want to turn detailed records into a clean report layout and there is exactly one value for each combination.


In [90]:
# Remove missing category combinations before pivoting back.
long_sales_clean = long_sales.dropna(subset=['Sales'])

# Pivot the long data back to Region × Category.
wide_again = long_sales_clean.pivot(
    index='Region', columns='Category', values='Sales'
)

# Display the result.

## 40. `pivot_table()` when aggregation is required

Use `pivot_table()` when duplicate combinations need summarisation.

**Practical use:** in retail data there are many transactions for the same Region and Category. `pivot_table()` can combine those transactions using `sum`, `mean`, `count` or another aggregation function to create a report.


In [91]:
# Create a pivot table directly from the transaction-level data.
pivot_sales = pd.pivot_table(
    wrangled,
    index='Region',
    columns='Category',
    values='Sales',
    aggfunc='sum',
    fill_value=0
)

# Display the pivot table.

## 41. Hierarchical columns with `pivot_table()`

`pivot_table()` can place more than one measure in the columns. The result is a **MultiIndex on the columns**.

This is useful when you want Sales, Profit and perhaps average Discount side-by-side for every Category.

In [92]:
# Create a pivot table with two measures: Sales and Profit.
category_report = pd.pivot_table(
    wrangled,
    index='Region',
    columns='Category',
    values=['Sales', 'Profit'],
    aggfunc='sum',
    fill_value=0
)

# Display the report.

# Inspect the column index names.

## 42. Working with hierarchical columns

When columns form a MultiIndex, selection can be done using tuples or by selecting one level.

For example, `(Sales, 'Bakery')` identifies one measure-category combination.

In [93]:
# Display the top level of the column MultiIndex.

# Display the second level of the column MultiIndex.

# Select one Sales column from category_report using a tuple.
# Use an available category name so the example is portable.

## 43. `stack()` – columns to rows

`stack()` moves one column level into the row index. It is useful when you need a more detailed, hierarchical row structure.

**Practical use:** a wide management report can be stacked back into a detailed analysis-friendly structure before further filtering or grouping.


In [94]:
# Stack the Category level of the simple pivot table.
stacked_sales = pivot_sales.stack()

# Display the stacked result.

# Inspect its index names.

## 44. `unstack()` – index levels to columns

`unstack()` performs the opposite type of movement: an index level is moved into the columns.

**Practical use:** this is useful when a grouped MultiIndex summary needs to become a wide report for presentation or comparison.


In [95]:
# Unstack the second index level from the Region-Category grouped summary.
# Store the result as unstacked_sales.

# Display the result.

## 45. Stacking and unstacking multiple levels

The operations can be combined when the data has more than one hierarchical level. This is useful for building detailed reports and then returning them to a tabular format.

In [96]:
# Create a Region-Category summary with two measures.
summary_multi = wrangled.groupby(['Region', 'Category'])[['Sales', 'Profit']].sum()

# Stack the measure columns so that Sales and Profit become another level.
# Store the result as stacked_multi.

# Use unstack() to bring Category back to columns.

## 46. Returning reshaped data to ordinary columns

After `stack()` or `unstack()`, the result may still have one or more index levels. `reset_index()` returns those levels to ordinary columns, which is often easier for exporting or merging.

In [97]:
# Reset the index of the final reshaped table.

# Display the column names.

## 47. Wide-to-long with `wide_to_long()`

`wide_to_long()` is useful when column names follow a repeated pattern such as `Sales_2015`, `Sales_2016`, and so on.

**Practical use:** this is common when a report stores one year per column but analysis requires one year per row.


In [98]:
# Build a small region-year wide table for the demonstration.
yearly = (
    wrangled.assign(Year=wrangled['Order Date'].dt.year)
           .groupby(['Region', 'Year'])['Sales']
           .sum()
           .unstack(fill_value=0)
           .reset_index()
)

# Rename the year columns using the pattern Sales_YYYY.
for col in [c for c in yearly.columns if isinstance(c, (int, np.integer))]:
    yearly = yearly.rename(columns={col: f'Sales_{col}'})

# Convert the repeated Sales_YYYY columns to long format with wide_to_long().
yearly_long = pd.wide_to_long(
    yearly, stubnames='Sales', i='Region', j='Year', sep='_', suffix='\d+'
).reset_index()

# Display the result.

<>:16: SyntaxWarning: invalid escape sequence '\d'
<>:16: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_7240/7850494.py:16: SyntaxWarning: invalid escape sequence '\d'
  yearly, stubnames='Sales', i='Region', j='Year', sep='_', suffix='\d+'


ValueError: Cannot mask with non-boolean array containing NA / NaN values

# Part G – Comparing the Main Reshaping Operations

| Operation | Typical direction | Best use |
|---|---|---|
| `melt()` | wide → long | turn many value columns into rows |
| `pivot()` | long → wide | reshape when each key combination is unique |
| `pivot_table()` | long → wide | reshape with aggregation |
| `stack()` | columns → index | create hierarchical row structure |
| `unstack()` | index → columns | spread index levels across columns |
| `wide_to_long()` | wide → long | repeated column-name patterns such as year suffixes |

A useful habit is to look at the **shape you have now** and the **shape you need next** before choosing the function.

# Part H – A Small End-to-End Wrangling Workflow

In a real project, the operations are usually connected rather than used separately.

For example, a sales analyst may start with transaction records, create a MultiIndex summary, attach a separate regional lookup, reshape the result into a management-friendly report, and finally verify the structure before sharing it.


## 48. Build a clean working table

In [ ]:
# Keep only the fields needed for the reporting task.
report_base = wrangled[['Order ID', 'Customer Name', 'Region', 'Category', 'Order Date', 'Sales', 'Profit']].copy()

# Create Year from Order Date.
report_base['Year'] = report_base['Order Date'].dt.year

# Check the result.

## 49. Create a hierarchical summary

In [ ]:
# Summarise Sales and Profit by Year, Region and Category.
report_summary = (
    report_base.groupby(['Year', 'Region', 'Category'])[['Sales', 'Profit']]
               .sum()
)

# Display the first few rows.

# Inspect the index levels.

## 50. Add a region lookup table

In [ ]:
region_lookup = pd.DataFrame({
    'Region': ['Central', 'East', 'North', 'South', 'West'],
    'Region_Group': ['Core', 'East Zone', 'North Zone', 'South Zone', 'West Zone']
})

# Reset the summary index and merge the region lookup.
report_flat = report_summary.reset_index()
report_enriched = pd.merge(report_flat, region_lookup, on='Region', how='left', validate='many_to_one')

# Display the enriched report.

## 51. Reshape the report for presentation

In [ ]:
# Create a wide report with Region as rows and Category as columns.
final_report = pd.pivot_table(
    report_enriched,
    index=['Year', 'Region_Group', 'Region'],
    columns='Category',
    values='Sales',
    aggfunc='sum',
    fill_value=0
)

# Display the final report.

# Inspect the index and column structure.

## 52. Verify the wrangled result

After several transformations, always perform a few structural checks. Look at the shape, missing values, duplicate rows, index structure and column structure.

In [ ]:
# Display the shape of final_report.

# Count missing values.

# Check for duplicate rows.

# Display the index names.

# Display the column names.

# Part I – Quick Checks

Before moving to the practice set, remember these quick checks:

```text
Need rows underneath each other?      → pd.concat(..., axis=0)
Need columns side-by-side?            → pd.concat(..., axis=1)
Need match using a key column?         → pd.merge()
Need match using existing indexes?     → merge(..., left_index=True, right_index=True) or join()
Need wide → long?                      → melt(), wide_to_long(), stack()
Need long → wide?                      → pivot(), pivot_table(), unstack()
Need multiple row/column levels?       → MultiIndex, stack(), unstack(), pivot_table()
```

# Practice Set

For each question, try the task yourself first. **No solutions are provided in this student practice section.** Use the worked examples above as a guide and check the structure of your result after each step.


## Question 1 – Create a MultiIndex

Create a new table with `Region` and `Category` as the MultiIndex. Keep `Sales` and `Profit` in the table.

In [ ]:
# Write your solution here.


## Question 2 – MultiIndex selection

Select all records belonging to the first available Region in the MultiIndex.

In [ ]:
# Write your solution here.


## Question 3 – Reorder and sort levels

Swap the two index levels and then sort the result by Category.

In [ ]:
# Write your solution here.


## Question 4 – Row-wise concatenation

Split the first 20 rows into two batches and combine them with `concat()`.

In [ ]:
# Write your solution here.


## Question 5 – Database-style left join

Create a customer sales summary and merge it with the **separate `customer_master` table** using a left join. Check that all customers from the sales summary are retained.


In [ ]:
# Write your solution here.


## Question 6 – Compare with a right join and different key names

First perform a right join so that all customers in `customer_master` are retained. Then rename `Customer Name` in the master table to `Customer` and repeat the merge using `left_on` and `right_on`.


In [ ]:
# Write your solution here.


## Question 7 – Merge on indexes

Create a Region-level Sales table and a separate Region-level target table, then combine them using `left_index=True` and `right_index=True`.


In [ ]:
# Write your solution here.


## Question 8 – Wide to long with `melt()`

Create a Region × Category Sales table and convert it to long format.

In [ ]:
# Write your solution here.


## Question 9 – Long to wide with `pivot_table()`

Use `pivot_table()` to create a Region × Category table showing total Profit.

In [ ]:
# Write your solution here.


## Question 10 – Hierarchical columns

Create a pivot table containing both Sales and Profit by Region and Category. Inspect its MultiIndex columns.

In [ ]:
# Write your solution here.


## Question 11 – Stack and unstack

Stack the Category level of a Region × Category Sales table and then unstack it back.

In [ ]:
# Write your solution here.


## Question 12 – Hierarchical reshaping

Create a Region-Category summary with Sales and Profit, then move Category from the index to the columns.

In [ ]:
# Write your solution here.


## Question 13 – Wide-to-long pattern

Create yearly regional sales columns such as `Sales_2015`, `Sales_2016`, and convert them back to long format with `wide_to_long()`.

In [ ]:
# Write your solution here.


## Question 14 – Mini case study

Build a report showing total Sales and Profit by Year and Region, add the region group, and create a wide Category report.

In [ ]:
# Write your solution here.


# Final Challenge – Build a Wrangled Reporting Dataset

Create your own report from the raw transaction data. The final result should:

1. use a MultiIndex at one intermediate stage,
2. combine at least two tables with `concat()` or `merge()`,
3. perform at least one index-based combination,
4. create a wide report with `pivot_table()`,
5. demonstrate either `stack()` or `unstack()`, and
6. finish with a verification section showing shape, missing values and index/column structure.

Try to keep the final report readable enough that another analyst can understand it without reading every intermediate cell.

In [ ]:
# Build your own end-to-end wrangling pipeline here.
# Keep the intermediate tables so that each transformation can be checked.

# Quick Reference

| Function | Purpose |
|---|---|
| `set_index()` | create index / MultiIndex from columns |
| `pd.MultiIndex.from_arrays()` | create a MultiIndex from arrays |
| `pd.MultiIndex.from_tuples()` | create a MultiIndex from tuples |
| `.loc[]` | label-based selection |
| `xs()` | select a cross-section from one index level |
| `swaplevel()` | exchange MultiIndex levels |
| `reorder_levels()` | change the level order |
| `sort_index()` | sort by index levels |
| `reset_index()` | move index levels back to columns |
| `pd.concat()` | combine tables along rows or columns |
| `pd.merge()` | database-style joins (`inner`, `left`, `right`, `outer`) |
| `DataFrame.join()` | convenient index-based join |
| `melt()` | wide → long |
| `pivot()` | long → wide without aggregation |
| `pivot_table()` | long → wide with aggregation |
| `stack()` | columns → index |
| `unstack()` | index → columns |
| `wide_to_long()` | repeated wide-column patterns → long |

### Key idea

**Data wrangling is mainly about changing structure without losing meaning.** Before choosing a function, ask: *What is the key? What should stay as rows? What should become columns? Do I need matching, stacking, or aggregation?*

## End of Practical

You have now worked through the main pandas operations used for hierarchical indexing, combining datasets, database-style joins, merging on indexes, reshaping, and pivoting. Keep the intermediate tables while practising—you will often learn more from comparing the structures before and after a transformation than from looking only at the final result.